# 06 - Test Evaluation and Temporal Smoothing


In [1]:
!uv pip install huggingface_hub webdataset librosa soundfile xgboost statsmodels pyarrow fastparquet

Using Python 3.12.3 environment at: /home/desan/Hackathons/octwave/.venv
Checked 8 packages in 89ms


In [2]:
# ── Section 0-B: imports ──────────────────────────────────────────────────────
import tarfile
import sys, os, re, json, io, time, logging, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scipy.signal as ss
import scipy.stats as st
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import soundfile as sf

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[WARN] XGBoost not available; falling back to HistGradientBoostingClassifier")

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

import webdataset as wds
from huggingface_hub import hf_hub_download, list_repo_files, snapshot_download

import joblib

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Python  : {sys.version}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"librosa : {librosa.__version__}")
print(f"sklearn : {__import__('sklearn').__version__}")
if HAS_XGB:
    print(f"xgboost : {xgb.__version__}")

Python  : 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
NumPy   : 2.5.3
Pandas  : 3.0.5
librosa : 1.0.0
sklearn : 1.9.0
xgboost : 3.4.0


In [3]:
# ── Section 0-C: global configuration ────────────────────────────────────────
# All tuneable parameters in one place.

RANDOM_SEED   = 42
np.random.seed(RANDOM_SEED)

# Dataset
HF_REPO       = "DesanSilva/sscc-compact-av"
HF_REPO_TYPE  = "dataset"
DATA_DIR      = Path("data/sscc_compact")
OUT_DIR       = Path("outputs")

# Signal parameters (from SSCC spec)
VIB_RATE_HZ       = 100_000     # original acquisition rate
VIB_STORED_HZ     = 10_000      # stored/downsampled rate (compact set)
AUD_RATE_HZ       = 44_100      # recorder (Zoom H5) rate
CLIP_DURATION_S   = 5.0
N_VIB_CHANNELS    = 4

# Feature extraction windows
WINDOW_SIZE_S = 1.0
HOP_SIZE_S    = 0.5

# Vibration frequency bands (Hz) - physically motivated for chain conveyor
VIB_BANDS = {
    "low":    (0,    500),
    "mid":    (500,  2000),
    "high":   (2000, 5000),
}

# Audio frequency bands (Hz)
AUD_BANDS = {
    "sub":    (0,    200),
    "low":    (200,  1000),
    "mid":    (1000, 4000),
    "high":   (4000, 8000),
    "air":    (8000, 16000),
}

# Envelope analysis bandpass (Hz) - mechanical fault region
ENV_BANDPASS_HZ = (500, 4000)

# MFCC
N_MFCC = 13

# Temporal smoothing
EMA_ALPHA     = 0.4
MAVG_WINDOW   = 3
NOM_N         = 3
NOM_M         = 5
PERSIST_K     = 3

# Split sizes
VALID_FRAC = 0.15
TEST_FRAC  = 0.15

# XGBoost / fallback
XGB_ROUNDS        = 500
XGB_EARLY_STOP    = 30
XGB_N_FOLDS       = 5

# Output subdirectories
for sub in ["eda", "features", "models", "metrics", "reports"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
log = logging.getLogger("sscc_poc")

# Plotting defaults
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
PALETTE = {"normal": "#2ecc71", "dry": "#e74c3c", "lean": "#e67e22",
           "loose": "#9b59b6", "screwdrop": "#3498db"}

print("Configuration complete.")

Configuration complete.


In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
# Use a clean, professional plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")



In [5]:
import pandas as pd
import joblib
meta_df = pd.read_parquet(OUT_DIR / '02_prep_state.parquet')
vib_clip_df = pd.read_parquet(OUT_DIR / 'features' / '03_vib_clip.parquet')
aud_clip_df = pd.read_parquet(OUT_DIR / 'features' / '03_aud_clip.parquet')
vib_win_df = pd.read_parquet(OUT_DIR / 'features' / '03_vib_win.parquet')
aud_win_df = pd.read_parquet(OUT_DIR / 'features' / '03_aud_win.parquet')
models = joblib.load(OUT_DIR / 'models' / '04_trained_models.pkl')
xgb_vib_bin = models['model_vib']
xgb_aud_bin = models['model_aud']
best_th_fused = models['best_th_fused']
splits = joblib.load(OUT_DIR / 'features' / '04_splits.pkl')
vib_tr, vib_va, vib_te = splits['vib_tr'], splits['vib_va'], splits['vib_te']
aud_tr, aud_va, aud_te = splits['aud_tr'], splits['aud_va'], splits['aud_te']
train_set, test_set, test_set = splits['train_set'], splits['test_set'], splits['test_set']
print('Evaluation data loaded.')


Evaluation data loaded.


### Loaded Helper Functions


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
import warnings; warnings.filterwarnings("ignore")
import logging
log = logging.getLogger("sscc_poc")

META_COLS = {"sample_id", "fault", "is_fault", "velocity", "source_index",
             "window_idx", "win_start_s", "win_end_s"}

def _feature_cols(df: pd.DataFrame) -> list:
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c not in META_COLS]


def clean_feature_df(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Remove NaN, inf, zero-variance, and near-constant features."""
    fc = _feature_cols(df)
    X = df[fc].copy()

    # Infinite → NaN → column mean imputation
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    n_inf = X.isnull().sum().sum()
    if n_inf:
        log.warning("%s: %d inf/NaN values imputed with column mean", name, n_inf)
    X = X.fillna(X.mean())

    # Zero variance
    zv_cols = X.columns[X.std() < 1e-10].tolist()
    if zv_cols:
        log.warning("%s: dropping %d zero-variance features", name, len(zv_cols))
        X.drop(columns=zv_cols, inplace=True)

    # Near-zero variance (std < 1% of range)
    near_zv = [c for c in X.columns if X[c].std() < 0.01 * (X[c].max() - X[c].min() + 1e-12)]
    if near_zv:
        log.info("%s: %d near-zero-variance features flagged (kept)", name, len(near_zv))

    print(f"{name}: {X.shape[1]} features remaining after QC")
    return pd.concat([df[list(META_COLS & set(df.columns))], X], axis=1)


vib_clip_df = clean_feature_df(vib_clip_df, "vibration")
aud_clip_df = clean_feature_df(aud_clip_df, "audio")

2026-09-19 12:52:47,231 [WARNING] vibration: dropping 3 zero-variance features


vibration: 885 features remaining after QC
audio: 600 features remaining after QC


In [7]:
import joblib
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
)
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

XGB_ROUNDS     = 500
XGB_EARLY_STOP = 30
XGB_N_FOLDS    = 5

def build_xgb_binary(n_pos: int, n_neg: int, seed: int = 42) -> object:
    scale_pos = n_neg / (n_pos + 1e-6)
    if HAS_XGB:
        return xgb.XGBClassifier(
            n_estimators=XGB_ROUNDS, learning_rate=0.05,
            max_depth=5, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos, eval_metric="logloss",
            early_stopping_rounds=XGB_EARLY_STOP,
            use_label_encoder=False, random_state=seed, n_jobs=-1,
        )
    return HistGradientBoostingClassifier(
        max_iter=XGB_ROUNDS, learning_rate=0.05,
        max_depth=5, random_state=seed,
        class_weight={0: 1, 1: scale_pos},
    )


def build_xgb_mc(n_classes: int, seed: int = 42) -> object:
    if HAS_XGB:
        return xgb.XGBClassifier(
            n_estimators=XGB_ROUNDS, learning_rate=0.05,
            max_depth=5, subsample=0.8, colsample_bytree=0.8,
            objective="multi:softprob", num_class=n_classes,
            eval_metric="mlogloss",
            early_stopping_rounds=XGB_EARLY_STOP,
            use_label_encoder=False, random_state=seed, n_jobs=-1,
        )
    from sklearn.ensemble import HistGradientBoostingClassifier
    return HistGradientBoostingClassifier(
        max_iter=XGB_ROUNDS, learning_rate=0.05,
        max_depth=5, random_state=seed,
    )


def train_binary_xgb(
    X_tr: np.ndarray, y_tr: np.ndarray,
    X_va: np.ndarray, y_va: np.ndarray,
    name: str,
) -> Tuple:
    """Train XGBoost binary classifier; return (model, evals_result)."""
    n_pos = int(y_tr.sum())
    n_neg = int((y_tr == 0).sum())
    clf = build_xgb_binary(n_pos, n_neg)
    if HAS_XGB:
        clf.fit(
            X_tr, y_tr,
            etest_set=[(X_tr, y_tr), (X_va, y_va)],
            verbose=False,
        )
        evals = clf.evals_result()
    else:
        clf.fit(X_tr, y_tr)
        evals = {}
    joblib.dump(clf, OUT_DIR / "models" / f"{name}.pkl")
    return clf, evals


def plot_training_curves(evals: dict, name: str) -> None:
    if not evals:
        print(f"[SKIP] No training curve data for {name}")
        return
    keys = list(evals.keys())  # e.g. ['validation_0', 'validation_1']
    metric = list(evals[keys[0]].keys())[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    labels = ["Train", "Validation"]
    for ki, key in enumerate(keys[:2]):
        vals = evals[key][metric]
        ax.plot(vals, label=labels[ki])
    ax.set_xlabel("Boosting round")
    ax.set_ylabel(metric)
    ax.set_title(f"Training curve - {name}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "eda" / f"training_curve_{name}.png", bbox_inches="tight")
    plt.show()


def evaluate_binary(
    clf, X: np.ndarray, y: np.ndarray,
    threshold: float = 0.5, name: str = "",
) -> dict:
    """Compute binary classification metrics."""
    prob = clf.predict_proba(X)[:, 1]
    pred = (prob >= threshold).astype(int)
    cm   = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    specificity = tn / (tn + fp + 1e-12)

    n_classes_in_y = len(np.unique(y))
    roc_auc = roc_auc_score(y, prob) if n_classes_in_y == 2 else float("nan")
    pr_auc  = average_precision_score(y, prob) if n_classes_in_y == 2 else float("nan")

    return {
        "name":        name,
        "threshold":   threshold,
        "accuracy":    accuracy_score(y, pred),
        "precision":   precision_score(y, pred, zero_division=0),
        "recall":      recall_score(y, pred, zero_division=0),
        "specificity": specificity,
        "f1":          f1_score(y, pred, zero_division=0),
        "roc_auc":     roc_auc,
        "pr_auc":      pr_auc,
        "confusion_matrix": cm,
        "prob":        prob,
        "pred":        pred,
    }


def evaluate_multiclass(
    clf, X: np.ndarray, y_str: np.ndarray, le,
    name: str = "",
) -> dict:
    """Multiclass metrics."""
    y = le.transform(y_str)
    prob = clf.predict_proba(X)
    pred_enc = clf.predict(X)
    pred_str = le.inverse_transform(pred_enc)
    classes  = le.classes_

    mc_roc = float("nan")
    if len(np.unique(y)) > 1:
        try:
            mc_roc = roc_auc_score(y, prob, multi_class="ovr", average="macro")
        except Exception:
            pass

    return {
        "name":            name,
        "accuracy":        accuracy_score(y, pred_enc),
        "macro_precision": precision_score(y, pred_enc, average="macro", zero_division=0),
        "macro_recall":    recall_score(y, pred_enc, average="macro", zero_division=0),
        "macro_f1":        f1_score(y, pred_enc, average="macro", zero_division=0),
        "weighted_f1":     f1_score(y, pred_enc, average="weighted", zero_division=0),
        "roc_auc_ovr":     mc_roc,
        "confusion_matrix":confusion_matrix(y, pred_enc),
        "classes":         classes,
        "pred_str":        pred_str,
        "y_str":           y_str,
        "prob":            prob,
    }


def plot_confusion(cm: np.ndarray, labels: list, title: str, out_name: str) -> None:
    fig, ax = plt.subplots(figsize=(max(4, len(labels)), max(3, len(labels))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "eda" / f"{out_name}.png", bbox_inches="tight")
    plt.show()


def plot_roc_pr(metrics: dict, title: str, out_name: str) -> None:
    y_true = metrics.get("y_true")
    prob   = metrics["prob"]
    if y_true is None or np.isnan(metrics["roc_auc"]):
        print(f"[SKIP ROC] {title}")
        return
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    fpr, tpr, _ = roc_curve(y_true, prob)
    axes[0].plot(fpr, tpr, lw=1.5, label=f"AUC={metrics['roc_auc']:.3f}")
    axes[0].plot([0,1],[0,1],"--",color="grey")
    axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
    axes[0].set_title(f"ROC - {title}"); axes[0].legend()

    prec, rec, _ = precision_recall_curve(y_true, prob)
    axes[1].plot(rec, prec, lw=1.5, label=f"AP={metrics['pr_auc']:.3f}")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title(f"PR - {title}"); axes[1].legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "eda" / f"roc_pr_{out_name}.png", bbox_inches="tight")
    plt.show()


print("Classifier and evaluation helpers defined.")

Classifier and evaluation helpers defined.


In [8]:
import joblib
import numpy as np
vib_scaler = joblib.load(OUT_DIR / 'models' / 'vib_scaler.pkl')
aud_scaler = joblib.load(OUT_DIR / 'models' / 'aud_scaler.pkl')
fused_scaler = joblib.load(OUT_DIR / 'models' / 'fused_scaler.pkl')
fold_models = joblib.load(OUT_DIR / 'models' / 'fused_detector_ensemble.pkl')

vib_fc = _feature_cols(vib_clip_df)
aud_fc = _feature_cols(aud_clip_df)

def _scale_vib(df: pd.DataFrame) -> np.ndarray:
    return vib_scaler.transform(df[vib_fc].fillna(0)).astype(np.float32)

def _scale_aud(df: pd.DataFrame) -> np.ndarray:
    return aud_scaler.transform(df[aud_fc].fillna(0)).astype(np.float32)

def build_fused_df(vib_df: pd.DataFrame, aud_df: pd.DataFrame) -> pd.DataFrame:
    vfc = [c for c in _feature_cols(vib_df)]
    afc = [c for c in _feature_cols(aud_df)]
    vib_sub = vib_df[['sample_id','fault','is_fault','velocity','source_index'] + vfc]
    aud_sub = aud_df[['sample_id'] + afc]
    fused = vib_sub.merge(aud_sub, on='sample_id', how='inner')
    if 'aud_rms_mean' in fused.columns and 'vib_ch1_rms_mean' in fused.columns:
        fused['cross_aud_vib_rms_ratio'] = (fused['aud_rms_mean'] / (fused['vib_ch1_rms_mean'] + 1e-12))
    return fused

def _scale_fused(df: pd.DataFrame) -> np.ndarray:
    fc = [c for c in _feature_cols(df)]
    return fused_scaler.transform(df[fc].fillna(0)).astype(np.float32)

def fused_predict_proba(X: np.ndarray) -> np.ndarray:
    return np.mean([m.predict_proba(X)[:,1] for m in fold_models], axis=0)


---
# 1. Temporal Smoothing

The fused classifier produces `p_fault(t)` for each sequential window within a clip. We implement four temporal decision strategies:

- **A - Moving average:** smooth probability over last k windows
- **B - EMA:** exponentially weighted moving average
- **C - N-of-M voting:** alarm if ≥ N of last M windows exceed threshold
- **D - Persistence:** alarm only after K consecutive windows exceed threshold

> **Important distinction:** These strategies reduce transient false alarms. They do **not** predict future faults - they stabilise the *current-state* decision.

In [9]:
# ── Section 18-A: build window-level fused probabilities ──────────────────────
# For temporal smoothing, we need probabilities at window granularity.
# We merge window-level vib and audio features, scale, and score.
def build_fused_window_df(vib_w: pd.DataFrame, aud_w: pd.DataFrame) -> pd.DataFrame:
    vfc = [c for c in _feature_cols(vib_w)]
    afc = [c for c in _feature_cols(aud_w)]
    vib_sub = vib_w[["sample_id","fault","is_fault","velocity",
                      "source_index","window_idx"] + vfc]
    aud_sub = aud_w[["sample_id","window_idx"] + afc]
    return vib_sub.merge(aud_sub, on=["sample_id","window_idx"], how="inner")
# We need to compute window-level fused features and score them
# Since the window-level dfs use the same feature names as clip-level (but not aggregated),
# we build a separate window-level scaler.
def _window_feature_cols(df: pd.DataFrame, exclude_set: set = None) -> list:
    exc = (exclude_set or set()) | META_COLS | {"window_idx", "win_start_s", "win_end_s"}
    return [c for c in df.select_dtypes(include=[np.number]).columns if c not in exc]
# Build window-level fused feature matrix (train split only for fitting scaler)
vib_win_tr = vib_win_df[vib_win_df["sample_id"].isin(train_set)]
aud_win_tr = aud_win_df[aud_win_df["sample_id"].isin(train_set)]
fused_win_tr = build_fused_window_df(vib_win_tr, aud_win_tr)
fused_win_fc = _window_feature_cols(fused_win_tr)
win_scaler = StandardScaler().fit(fused_win_tr[fused_win_fc].fillna(0))
joblib.dump(win_scaler, OUT_DIR / "models" / "fused_window_scaler.pkl")
import xgboost as xgb
print('Training window-level model...')
X_win_tr = win_scaler.transform(fused_win_tr[fused_win_fc].fillna(0)).astype(np.float32)
y_win_tr = np.array(fused_win_tr['is_fault'])
win_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1)
win_model.fit(X_win_tr, y_win_tr)
print('Window-level model trained.')
def score_windows(sample_ids: list, vib_w: pd.DataFrame, aud_w: pd.DataFrame) -> pd.DataFrame:
    """Return window-level probability predictions for given sample_ids."""
    vib_sub = vib_w[vib_w["sample_id"].isin(sample_ids)]
    aud_sub = aud_w[aud_w["sample_id"].isin(sample_ids)]
    fused_w = build_fused_window_df(vib_sub, aud_sub)
    X_w = win_scaler.transform(fused_w[fused_win_fc].fillna(0)).astype(np.float32)
    probs = win_model.predict_proba(X_w)[:, 1]
    fused_w = fused_w.copy()
    fused_w["p_fault"] = probs
    return fused_w
print("Window-level scoring pipeline ready.")


Training window-level model...
Window-level model trained.
Window-level scoring pipeline ready.


In [10]:
# ── Section 18-B: smoothing strategies ───────────────────────────────────────

EMA_ALPHA  = 0.4
MAVG_WIN   = 3
NOM_N      = 3
NOM_M      = 5
PERSIST_K  = 3

def smooth_mavg(probs: np.ndarray, k: int = MAVG_WIN) -> np.ndarray:
    """Moving average over last k windows."""
    out = np.zeros_like(probs)
    for i in range(len(probs)):
        out[i] = probs[max(0, i-k+1):i+1].mean()
    return out


def smooth_ema(probs: np.ndarray, alpha: float = EMA_ALPHA) -> np.ndarray:
    """Exponential moving average."""
    out = np.zeros_like(probs)
    out[0] = probs[0]
    for i in range(1, len(probs)):
        out[i] = alpha * probs[i] + (1 - alpha) * out[i-1]
    return out


def decide_nom(probs: np.ndarray, threshold: float,
                N: int = NOM_N, M: int = NOM_M) -> np.ndarray:
    """N-of-M: alarm if ≥ N of last M windows exceed threshold."""
    decisions = (probs >= threshold).astype(int)
    out = np.zeros_like(decisions)
    for i in range(len(decisions)):
        window = decisions[max(0, i-M+1):i+1]
        out[i] = int(window.sum() >= N)
    return out


def decide_persist(probs: np.ndarray, threshold: float,
                    K: int = PERSIST_K) -> np.ndarray:
    """Persistence: alarm only after K consecutive windows above threshold."""
    decisions = (probs >= threshold).astype(int)
    out = np.zeros_like(decisions)
    consec = 0
    for i, d in enumerate(decisions):
        consec = consec + 1 if d else 0
        out[i] = int(consec >= K)
    return out


def evaluate_temporal(
    smooth_probs: np.ndarray, y_true: np.ndarray,
    threshold: float, strategy_name: str,
    hard_decisions: np.ndarray = None,
) -> dict:
    """Compute F1, recall, specificity, false-alarm count, missed-fault count, latency proxy."""
    if hard_decisions is None:
        hard_decisions = (smooth_probs >= threshold).astype(int)
    cm = confusion_matrix(y_true, hard_decisions)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)
    # Latency: average index of first correct fault alarm (within each fault block)
    return {
        "strategy":    strategy_name,
        "recall":      tp / (tp + fn + 1e-12),
        "f1":          f1_score(y_true, hard_decisions, zero_division=0),
        "specificity": tn / (tn + fp + 1e-12),
        "false_alarms":fp,
        "missed":      fn,
    }


print("Temporal smoothing functions defined.")

Temporal smoothing functions defined.


In [11]:
# ── Section 18-C: evaluate smoothing strategies on test windows ──────────

test_win_scored = score_windows(list(test_set), vib_win_df, aud_win_df)
test_win_scored = test_win_scored.sort_values(["sample_id", "window_idx"]).reset_index(drop=True)

p_raw = test_win_scored["p_fault"].to_numpy()
y_win = test_win_scored["is_fault"].to_numpy()

th = best_th_fused
p_mavg   = smooth_mavg(p_raw)
p_ema    = smooth_ema(p_raw)
nom_dec  = decide_nom(p_raw, th)
pers_dec = decide_persist(p_raw, th)

temporal_results = [
    evaluate_temporal(p_raw,   y_win, th, "Raw classifier"),
    evaluate_temporal(p_mavg,  y_win, th, f"Moving avg (k={MAVG_WIN})"),
    evaluate_temporal(p_ema,   y_win, th, f"EMA (α={EMA_ALPHA})"),
    evaluate_temporal(None, y_win, th, f"N-of-M ({NOM_N}/{NOM_M})", hard_decisions=nom_dec),
    evaluate_temporal(None, y_win, th, f"Persistence (K={PERSIST_K})", hard_decisions=pers_dec),
]

temp_df = pd.DataFrame(temporal_results)[["strategy","recall","f1","specificity","false_alarms","missed"]]
temp_df = temp_df.round(3)
print("\nTemporal smoothing comparison (test windows):")
print(temp_df.to_string(index=False))

temp_df.to_csv(OUT_DIR / "metrics" / "temporal_comparison.csv", index=False)


Temporal smoothing comparison (test windows):
         strategy  recall    f1  specificity  false_alarms  missed
   Raw classifier    1.00 1.000         1.00             0       0
 Moving avg (k=3)    1.00 0.995         0.99             2       0
      EMA (α=0.4)    1.00 0.990         0.98             4       0
     N-of-M (3/5)    0.99 0.990         0.99             2       2
Persistence (K=3)    0.99 0.995         1.00             0       2
